In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
tokenizer = Tokenizer(char_level=False)

In [ ]:
faq = """The sun rises early in the morning.
I wake up and stretch my arms.
First, I make a hot cup of coffee.
The smell of the beans fills the kitchen.
I eat a piece of toasted bread with butter.
After breakfast, I put on my running shoes.
I walk outside to breathe the fresh air.
The birds are singing in the tall trees.
I see a small dog running in the park.
My neighbor waves hello from across the street.
I walk back home to start my work.
I sit at my desk and turn on the computer.
The screen glows brightly in the dim room.
I type quickly on the keyboard to finish my tasks.
When I am tired, I take a short break.
I drink a glass of cold water.
Life is simple and the day is beautiful."""

In [ ]:
tokenizer.fit_on_texts([faq])

In [ ]:
word_index = tokenizer.word_index

In [ ]:
input_sequences = []
for sentence in faq.split('\n'):
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1, len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])


In [ ]:
input_sequences

In [ ]:
max_len = max(len(x) for x in input_sequences)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequence = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

In [ ]:
x = padded_input_sequence[:, :-1]

In [ ]:
y = padded_input_sequence[:, -1]

In [ ]:
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes = 92)  # one hot encoding for classification

In [ ]:
x.shape, y.shape

((123, 9), (123, 92))

#### Architecture

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
import numpy as np

In [ ]:
model = Sequential()
model.add(Embedding(92, 100))
model.add(LSTM(150))
model.add(Dense(92, activation='softmax'))
model.build((None, x.shape[1])) # Explicitly build the model with correct input shape

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_16 (Embedding)        │ (None, 9, 100)         │         9,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_20 (LSTM)                  │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 92)             │        13,892 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 173,692 (678.48 KB)

 Trainable params: 173,692 (678.48 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(x, y, epochs=100, validation_data=(x, y))

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 124ms/step - accuracy: 0.0488 - loss: 4.5218 - val_accuracy: 0.0976 - val_loss: 4.5079
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.0976 - loss: 4.5037 - val_accuracy: 0.0894 - val_loss: 4.4888
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.0894 - loss: 4.4809 - val_accuracy: 0.0894 - val_loss: 4.4546
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0894 - loss: 4.4357 - val_accuracy: 0.0894 - val_loss: 4.3774
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0894 - loss: 4.3345 - val_accuracy: 0.0894 - val_loss: 4.2416
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0894 - loss: 4.2704 - val_accuracy: 0.0894 - val_loss: 4.2266
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0894 - loss: 4.2111 - val_accuracy: 0.0894 - val_loss: 4.1705
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1138 - loss: 4.1871 - val_accuracy: 0.1301 - val_loss

In [ ]:
text = 'can'

for i in range(5):
  #token
  input_text_token = tokenizer.texts_to_sequences([text])[0]
  #padding
  padded_token_text = pad_sequences([input_text_token], maxlen=max_len, padding='pre')
  #predict
  output_val = np.argmax(model.predict(padded_token_text))

  for word, index in word_index.items():
    if index == output_val:
      text = text + ' ' + word
      print(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
can neighbor
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
can neighbor waves
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
can neighbor waves hello
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
can neighbor waves hello hello
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
can neighbor waves hello hello from
